# Laboratorio 2 — Del análisis estadístico al primer modelo

En el **Laboratorio 1** respondimos una pregunta puntual con una prueba de
hipótesis: ¿la nota promedio de quienes asisten es distinta de la de quienes
no asisten? Ese análisis vive en `academic_analysis.analysis` y sus
resultados ya quedaron guardados en `results/`.

Aquí damos un paso más: entrenamos un **primer modelo predictivo** —una
regresión que intenta predecir la nota a partir de la asistencia, el curso y
el semestre— y con él ilustramos dos ideas de la Unidad 4:

- **Reproducibilidad** (Capítulo 1 y 2): el modelo depende de una versión
  concreta de los datos, el código y una semilla aleatoria fija
  (`random_state = 42`).
- **Seguimiento de experimentos** (Capítulo 3): cada entrenamiento queda
  registrado en **MLflow** con sus parámetros, métricas y el modelo como
  artefacto, en vez de vivir únicamente en la memoria de quien lo entrenó.

Este notebook es para **explorar y explicar**. El proceso "real" y
repetible —el que se ejecuta cada vez que llega una nueva ingesta— vive en
los scripts de `src/academic_analysis/` (`train.py`, `analysis.py`, etc.).
Aquí simplemente reutilizamos esas mismas funciones para poder ver, paso a
paso, qué hace cada una.

In [1]:
import json

import pandas as pd

from academic_analysis.train import (
    ROOT,
    PREPARED_DIR,
    build_training_frame,
    load_manifest,
    train_regression_model,
)

DATASET_ID = "ING-20260910-101728"

manifest_path, manifest = load_manifest(DATASET_ID)
manifest["status"]

'trained'

## 1. Los datos que vamos a usar

Partimos del dataset **preparado** (`data/prepared/`), no del crudo. Ya pasó
por validación de esquema y limpieza (`academic_analysis.validate` y
`academic_analysis.prepare`), y el manifiesto (`metadata/`) guarda el
SHA-256 de ese archivo. Por eso `train.py` puede comprobar, antes de
entrenar, que nadie modificó el archivo por fuera del pipeline.

In [2]:
df = pd.read_csv(PREPARED_DIR / manifest["preparation"]["output"])
print(df.shape)
df.head()

(6231, 6)


,student_id,Curso,Semestre,Nota Curso,n_asistencias,asistio
0,017fa3271c12e0358caa0b3a,Cálculo Diferencial,2025-2,NaN,0,False
1,01c00cf8e4ca011bb5fd33a8,Cálculo Diferencial,2025-2,3.75,0,False
2,021c8ee196052cbdf6918d8d,Cálculo Diferencial,2025-2,4.54,4,True
3,02630362cdcfe8de2f79f265,Cálculo Diferencial,2025-2,4.20,0,False
4,073a35b3bf7f280c07e7dd3d,Cálculo Diferencial,2025-2,2.17,0,False


## 2. Lo que ya sabíamos: el resultado del Laboratorio 1

Antes de entrenar nada, recordemos qué encontró la prueba de hipótesis por
curso. Reutilizamos el resultado ya calculado en `results/`, no lo volvemos
a calcular aquí.

In [3]:
with open(ROOT / "results" / f"analysis_{DATASET_ID}.json", encoding="utf-8") as f:
    analysis = json.load(f)

courses = pd.DataFrame(analysis["courses"])
courses[
    ["Curso", "media_asisten", "media_no_asisten", "p_value", "significativo"]
].sort_values("p_value")

,Curso,media_asisten,media_no_asisten,p_value,significativo
11,Geometría Analítica,3.486759,3.040061,5.939488e-07,True
0,Cálculo Diferencial,3.353692,2.942937,1.187273e-05,True
1,Cálculo Integral,3.379402,3.101805,2.655943e-04,True
8,Fundamentos de Química,3.559643,2.980931,3.285310e-04,True
12,Álgebra Lineal,3.594472,3.424753,7.086421e-02,False
9,Física Mecánica,3.234091,3.168471,4.052789e-01,False
3,Ecuaciones Diferenciales,3.574051,3.687013,4.386655e-01,False
7,Fundamentos Matemáticos,3.682000,3.766278,4.894619e-01,False
4,Electricidad y Magnetismo,3.397000,3.347907,5.555848e-01,False
2,Cálculo Vectorial,3.040484,3.036017,9.647271e-01,False


Solo en unos pocos cursos la diferencia es estadísticamente significativa.
Eso ya nos advierte algo: la asistencia por sí sola probablemente **no va a
ser un predictor muy fuerte** de la nota. Vale la pena confirmarlo con un
modelo real en vez de suponerlo.

## 3. De "comparar dos grupos" a "predecir un número"

La prueba de hipótesis compara dos promedios. Un modelo de regresión es más
ambicioso: para cada estudiante, intenta predecir la nota exacta a partir de
sus variables. Usamos:

- `n_asistencias` (numérica)
- `Curso` (categórica → codificada con *one-hot encoding*)
- `Semestre` (categórica → codificada con *one-hot encoding*)

y como variable objetivo, `Nota Curso` (solo en las filas donde existe).

In [4]:
x, y = build_training_frame(df)
print("Filas con nota disponible:", len(y))
print("Variables después de codificar:", x.shape[1])
x.head()

Filas con nota disponible: 3041
Variables después de codificar: 13


,n_asistencias,Curso_Cálculo Integral,Curso_Cálculo Vectorial,Curso_Ecuaciones Diferenciales,Curso_Electricidad y Magnetismo,Curso_Fundamentos Matemáticos,Curso_Fundamentos de Química,Curso_Física Mecánica,Curso_Física Moderna,Curso_Geometría Analítica,Curso_Álgebra Lineal,Semestre_2026-1,Semestre_2026-2
1,0,False,False,False,False,False,False,False,False,False,False,False,False
2,4,False,False,False,False,False,False,False,False,False,False,False,False
3,0,False,False,False,False,False,False,False,False,False,False,False,False
4,0,False,False,False,False,False,False,False,False,False,False,False,False
8,2,False,False,False,False,False,False,False,False,False,False,False,False


## 4. Entrenar, con seguimiento en MLflow

`train_regression_model` hace lo mismo que `academic_analysis.train`
ejecutado desde la terminal: separa entrenamiento/prueba con una semilla
fija, entrena una `LinearRegression`, calcula métricas, y registra todo en
MLflow (parámetros + métricas + el modelo como artefacto). Además actualiza
`metadata/<dataset_id>.json` con un resumen versionable en Git.

In [5]:
model_output, training = train_regression_model(DATASET_ID)
training["metrics"]

C:\Users\000010478\Downloads\analisis_datos_academicos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/09/25 16:20:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'rmse': 0.8321884656826379, 'r2': 0.08351088462559308}

## 5. Interpretar el resultado (para toda la audiencia, no solo para quien programa)

- **RMSE** (~0.8, sobre una escala de nota de 0 a 5): en promedio, el modelo
  se equivoca por menos de un punto.
- **R²** (cercano a 0): la asistencia, el curso y el semestre explican una
  parte muy pequeña de por qué varía la nota. Hay muchos otros factores que
  el modelo no está viendo.

No es un mal resultado por ser "bajo": es un resultado **honesto**. Este
modelo es una primera aproximación (*baseline*) que sirve como punto de
comparación para los siguientes. Lo importante en esta etapa no es tener un
modelo excelente, sino tener un **proceso confiable** para entrenar,
medir y registrar cualquier modelo que construyamos después.

In [6]:
print("MLflow run_id      :", training["mlflow"]["run_id"])
print("MLflow experiment  :", training["mlflow"]["experiment_name"])
print("Tracking URI       :", training["mlflow"]["tracking_uri"])
print()
print("Para explorar todos los experimentos registrados:")
print(f"  uv run mlflow ui --backend-store-uri {training['mlflow']['tracking_uri']}")

MLflow run_id      : 9a8b605755694baca95794be07a02037
MLflow experiment  : nota_curso_regresion
Tracking URI       : sqlite:///C:\Users\000010478\Downloads\analisis_datos_academicos\mlflow.db

Para explorar todos los experimentos registrados:
  uv run mlflow ui --backend-store-uri sqlite:///C:\Users\000010478\Downloads\analisis_datos_academicos\mlflow.db


## 6. ¿Qué se versiona y dónde?

| Elemento | ¿Dónde vive? | ¿Se versiona en Git? |
|---|---|---|
| Datos preparados | `data/prepared/` | Sí |
| Código de entrenamiento | `src/academic_analysis/train.py` | Sí |
| Resumen del experimento (parámetros/métricas/run_id) | `metadata/<dataset_id>.json` | Sí |
| Modelo entrenado (`.joblib`) | `models/` | Sí |
| Historial completo de experimentos MLflow | `mlruns/`, `mlflow.db` | **No** — se reconstruye ejecutando `train.py` |

La idea, siguiendo la Unidad 4, no es guardar *todo* en Git. Es guardar lo
mínimo necesario para poder **reconstruir** el resto: con el código, los
datos y la semilla, `train.py` vuelve a producir un modelo equivalente en
cualquier computador.

## 7. Próximos pasos (aún por definir)

Este es un primer modelo, no el final. Los pasos naturales que siguen —y que
se irán decidiendo etapa por etapa— son, por ejemplo:

- Un segundo modelo, esta vez de **clasificación** (¿el estudiante
  aprueba o no?), para tener algo más "accionable".
- Comparar ambos modelos dentro de MLflow y decidir con qué criterio se
  elegiría un **modelo candidato**.
- Más adelante, decidir si tiene sentido exponer el mejor modelo como un
  servicio (Capítulo 4 de la Unidad 4).